# Visualizing `PitchUNet`

Inspect a trained (or freshly-initialized) `PitchUNet` — its layers, parameter
counts, weight distributions, and individual weight tensors rendered as heatmap
images with [Altair](https://altair-viz.github.io/).

**Set the checkpoint name in the config cell below.** Leave it empty to inspect a
randomly-initialized model.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import altair as alt

from model import PitchUNet, count_params

# Heatmaps emit one row per weight cell, which can exceed Altair's default cap.
alt.data_transformers.disable_max_rows()

## 1. Config — set your checkpoint here

In [ ]:
# ── USER CONFIG ───────────────────────────────────────────────────────────
# Name/path of the saved checkpoint to load.
# Checkpoints are saved as {"model": state_dict, ...} (see train.py),
# e.g. "final.pt" or "checkpoints/step_50000.pt".
# Leave as "" to inspect a fresh, randomly-initialized PitchUNet().
CKPT_PATH = ""

DEVICE = "cpu"

In [ ]:
model = PitchUNet().to(DEVICE)

if CKPT_PATH:
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"Loaded weights from: {CKPT_PATH}")
    if missing:
        print(f"  ! missing keys:    {len(missing)}")
    if unexpected:
        print(f"  ! unexpected keys: {len(unexpected)}")
else:
    print("No CKPT_PATH set — inspecting a freshly-initialized PitchUNet().")

model.eval()
print(f"Total trainable params: {count_params(model) / 1e6:.2f} M")

## 2. Layer overview

Flatten every named parameter tensor into a table with its shape, parameter
count, and basic statistics.

In [ ]:
rows = []
for name, p in model.named_parameters():
    rows.append(
        {
            "param": name,
            "module": name.split(".")[0],
            "shape": "×".join(map(str, tuple(p.shape))),
            "n_params": p.numel(),
            "ndim": p.dim(),
            "mean": float(p.detach().mean()),
            "std": 0.0 if p.numel() < 2 else float(p.detach().std()),
        }
    )

params_df = pd.DataFrame(rows)
print(f"{len(params_df)} weight tensors across {params_df['module'].nunique()} top-level modules")
params_df.head(12)

### Parameters per top-level module

In [ ]:
by_module = (
    params_df.groupby("module", as_index=False)["n_params"].sum()
    .sort_values("n_params", ascending=False)
)

alt.Chart(by_module).mark_bar().encode(
    x=alt.X("n_params:Q", title="parameter count"),
    y=alt.Y("module:N", sort="-x", title=None),
    color=alt.Color("module:N", legend=None),
    tooltip=["module:N", alt.Tooltip("n_params:Q", format=",")],
).properties(width=620, height=320, title="Parameters per top-level module")

### The 30 largest weight tensors

In [ ]:
top_n = params_df.nlargest(30, "n_params")

alt.Chart(top_n).mark_bar().encode(
    x=alt.X("n_params:Q", title="parameter count"),
    y=alt.Y("param:N", sort="-x", title=None),
    color=alt.Color("module:N", title="module"),
    tooltip=["param:N", "shape:N", alt.Tooltip("n_params:Q", format=",")],
).properties(width=620, height=620, title="Top 30 largest weight tensors")

## 3. Weight distributions

Per-module histograms of weight values. Note several layers are **zero-initialized
by design** (`FiLM.proj`, `out_conv`, and the attention positional embeddings), so
on a fresh model those distributions collapse to a single spike at 0.

In [ ]:
def weight_hist(values, bins=60):
    counts, edges = np.histogram(values, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    return pd.DataFrame({"value": centers, "count": counts})


hist_parts = []
for top in params_df["module"].unique():
    vals = np.concatenate(
        [
            p.detach().cpu().numpy().ravel()
            for n, p in model.named_parameters()
            if n.split(".")[0] == top
        ]
    )
    h = weight_hist(vals)
    h["module"] = top
    hist_parts.append(h)

hist_df = pd.concat(hist_parts, ignore_index=True)

alt.Chart(hist_df).mark_area(interpolate="step", opacity=0.85).encode(
    x=alt.X("value:Q", title="weight value"),
    y=alt.Y("count:Q", title="count"),
    color=alt.Color("module:N", legend=None),
    tooltip=["module:N", "value:Q", "count:Q"],
).properties(width=190, height=130).facet(
    facet=alt.Facet("module:N", title=None), columns=4
).resolve_scale(x="independent", y="independent")

### Mean ± std per tensor (top 40 by size)

In [ ]:
stat_df = params_df.nlargest(40, "n_params").copy()
stat_df["lo"] = stat_df["mean"] - stat_df["std"]
stat_df["hi"] = stat_df["mean"] + stat_df["std"]

y = alt.Y("param:N", sort=alt.EncodingSortField("n_params", order="descending"), title=None)
rule = alt.Chart(stat_df).mark_rule(color="#888").encode(
    y=y, x=alt.X("lo:Q", title="weight value"), x2="hi:Q"
)
point = alt.Chart(stat_df).mark_point(filled=True, size=45, color="black").encode(
    y=y, x="mean:Q", tooltip=["param:N", "mean:Q", "std:Q"]
)
(rule + point).properties(width=620, height=600, title="weight mean ± 1 std per tensor")

## 4. Weight tensors as heatmap images

Each weight tensor is flattened to 2D and drawn as a heatmap (diverging color
scale centered at 0). Conv kernels `[out, in, kh, kw]` become `[out, in·kh·kw]`;
large tensors are strided down so the image stays readable.

In [ ]:
_PARAMS = dict(model.named_parameters())


def to_2d(t: torch.Tensor) -> np.ndarray:
    """Flatten a weight tensor to a 2D array for display."""
    t = t.detach().cpu()
    if t.dim() == 1:
        t = t.unsqueeze(0)
    elif t.dim() > 2:
        t = t.reshape(t.shape[0], -1)
    return t.numpy()


def _downsample(a: np.ndarray, max_side: int) -> np.ndarray:
    h, w = a.shape
    return a[:: max(1, h // max_side), :: max(1, w // max_side)]


def heatmap(name: str, max_side: int = 160):
    """Render a named weight tensor as an Altair heatmap image."""
    p = _PARAMS[name]
    a = _downsample(to_2d(p), max_side)
    h, w = a.shape
    df = pd.DataFrame(
        {
            "row": np.repeat(np.arange(h), w),
            "col": np.tile(np.arange(w), h),
            "weight": a.ravel(),
        }
    )
    lim = float(np.abs(a).max()) or 1.0
    return (
        alt.Chart(df)
        .mark_rect()
        .encode(
            x=alt.X("col:O", axis=None),
            y=alt.Y("row:O", axis=None),
            color=alt.Color(
                "weight:Q",
                scale=alt.Scale(scheme="blueorange", domain=[-lim, lim]),
                title="weight",
            ),
            tooltip=["row:O", "col:O", alt.Tooltip("weight:Q", format=".4f")],
        )
        .properties(
            width=min(6 * w, 460),
            height=min(6 * h, 460),
            title=f"{name}  {tuple(p.shape)}",
        )
    )


# All layer names available for inspection:
list(_PARAMS.keys())

### The input stem convolution

In [ ]:
heatmap("stem.weight")

### A gallery of representative layers

In [ ]:
gallery = [
    "shift_mlp.encoder.0.weight",   # shift conditioning MLP
    "f0_enc.encoder.0.weight",      # f0 encoder (1D conv)
    "enc_blocks.0.conv1.weight",    # first encoder ResBlock
    "bot1.conv1.weight",            # bottleneck
    "attn.qkv.weight",              # self-attention qkv projection
    "dec_blocks.0.conv1.weight",    # first decoder ResBlock
    "out_conv.weight",              # output head (zero-initialized!)
]
gallery = [n for n in gallery if n in _PARAMS]

charts = [heatmap(n) for n in gallery]
rows_of = [
    alt.hconcat(*charts[i : i + 2]) for i in range(0, len(charts), 2)
]
alt.vconcat(*rows_of).resolve_scale(color="independent")

### Inspect any layer

Change the name to any entry printed by `list(_PARAMS.keys())` above.

In [ ]:
heatmap("enc_blocks.2.conv1.weight")